# 03 Exploratory data analysis

Use the fixed training split and the seven current-event fields approved in the [prediction design](../reports/prediction_design.md). Read the binary ground-truth target separately. Each row is a transaction, and events in the same batch share a label. The comparisons below describe this historical training sample. The timestamp is used for time plots and split checks. Conditional catalogue and batch fields are left out because scoring-time versions have yet to be verified.

In [ ]:
from hashlib import file_digest
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import seaborn as sns
from IPython.display import display

get_ipython().run_line_magic("matplotlib", "inline")
sns.set_theme(style="whitegrid")
current = Path.cwd()
root = next(path for path in (current, *current.parents) if (path / "AGENTS.md").is_file())
processed = root / "processed"
reports = root / "reports"
boundaries = json.loads((processed / "split_boundaries.json").read_text(encoding="utf-8"))
manifest_path = processed / boundaries["assignment_file"]
source_path = root / "Phase1_Dataset/dataset/transactions.parquet"
target_path = processed / "transaction_target.parquet"

for path, expected in [
    (manifest_path, boundaries["assignment_sha256"]),
    (source_path, boundaries["input_sha256"][source_path.relative_to(root).as_posix()]),
    (target_path, boundaries["input_sha256"][target_path.relative_to(root).as_posix()]),
]:
    with path.open("rb") as source:
        assert file_digest(source, "sha256").hexdigest() == expected, path

key = boundaries["transaction_key"]
clock = boundaries["prediction_timestamp"]
target = boundaries["target_column"]
audit = pd.read_csv(reports / "feature_audit.csv")
eligible = audit.loc[
    audit["table"].eq("transactions") & audit["category"].eq("Safe input features")
    & audit["planned_use"].isin(["current_event_input", "current_event_input_and_key"]),
    "column",
].tolist()
numeric = ["humidity", "weight", "battery_level"]
categorical = ["event_type", "leg", "rfid_scanned", "door_open"]
assert set(eligible) == set(numeric + categorical)
manifest = pd.read_parquet(manifest_path, columns=[*key, clock], filters=[("split", "=", "train")])
train_batch_ids = manifest["batch_id"].unique().tolist()
train_filter = [("batch_id", "in", train_batch_ids)]
# The split keeps each batch intact. Project approved fields and filter to the training batches on read.
event_columns = list(dict.fromkeys(key + eligible))
events = pd.read_parquet(source_path, columns=event_columns, filters=train_filter)
labels = pd.read_parquet(target_path, columns=[*key, target], filters=train_filter)
assert len(events) == len(labels) == len(manifest) == boundaries["split_summary"]["train"]["rows"]
for frame in (events, labels, manifest):
    assert frame[key].notna().all().all() and not frame.duplicated(key).any()

train = manifest.merge(events, on=key, validate="one_to_one")
train = train.merge(labels, on=key, validate="one_to_one")
assert len(train) == len(manifest) and train[clock].lt(pd.Timestamp(boundaries["validation_start_inclusive"])).all()
assert train[target].notna().all() and set(train[target].unique()) == {0, 1}
assert train.groupby("batch_id")[target].nunique().eq(1).all()
del manifest, events, labels
print(f"Verified {len(train):,} training transactions from {len(train_batch_ids):,} batches.")

## Target and missing values

The first chart counts genuine and fraudulent transactions. The shorter bar shows the minority class. The second chart shows the percentage of missing values for each eligible field, the event timestamp and the target. A bar at zero means every training row had a value in that field.

The training run found 10.01% fraudulent transactions and zero missing values in the plotted fields.

In [ ]:
target_counts = train[target].value_counts().sort_index().rename(index={0: "Genuine", 1: "Fraud"})
fraud_rate = train[target].mean() * 100
batch_labels = train.groupby("batch_id")[target].first()
missing = train[[*eligible, clock, target]].isna().sum().sort_values(ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3), layout="constrained")
sns.barplot(x=target_counts.index, y=target_counts.values, ax=axes[0], color="#36688d")
axes[0].bar_label(axes[0].containers[0], fmt=lambda value: f"{value:,.0f}", padding=4)
axes[0].set(title="Training target", ylabel="Transactions", ylim=(0, target_counts.max() * 1.15))
missing_pct = 100 * missing / len(train)
sns.barplot(y=missing.index, x=missing_pct.values, ax=axes[1], color="#36688d")
axes[1].bar_label(axes[1].containers[0], fmt="%.2f%%", padding=4, fontsize=8)
axes[1].set(title="Missing values", xlabel="Share of training rows (%)", xlim=(0, max(0.25, missing_pct.max() * 1.3)))
plt.show()
display(pd.DataFrame({"missing_rows": missing, "missing_pct": missing_pct}))

## Numerical distributions

The histograms count humidity, weight, and battery readings across value ranges. Look at where most readings fall and how far the tails extend. Battery uses an integer-centred bin for each observed level. The summary table reports the actual observed limits and central values.

Humidity and battery readings spread fairly evenly across their observed ranges. Weight readings cluster around 10.0.

In [ ]:
numeric_summary = train[numeric].describe(percentiles=[.01, .25, .5, .75, .99]).T
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), layout="constrained")
for field, ax in zip(numeric, axes):
    bins = np.arange(train[field].min() - .5, train[field].max() + 1, 1) if field == "battery_level" else 40
    sns.histplot(train[field], bins=bins, color="#36688d", ax=ax)
    ax.set(title=field, xlabel="Recorded value", ylabel="Transactions")
plt.show()
display(numeric_summary.round(3))

## Categorical distributions and fraud

Each row of the figure shows one field. The left chart counts the transactions in each category. The right chart shows the share of those transactions labelled fraud. The dashed line is the overall training fraud rate. Always check the counts alongside the rates. Several rows can carry the same batch label.

Each event type had 103,942 transactions and the same fraud share. Later legs had fewer transactions. An RFID scan was recorded and the door stayed closed on most rows.

In [ ]:
category_stats = {}
fig, axes = plt.subplots(len(categorical), 2, figsize=(12, 12), layout="constrained")
for field, (volume_ax, rate_ax) in zip(categorical, axes):
    stats = train.groupby(field, dropna=False, sort=True)[target].agg(rows="size", fraud="sum")
    stats["rate_pct"] = 100 * stats["fraud"] / stats["rows"]
    category_stats[field] = stats
    names = stats.index.astype(str)
    volume_ax.barh(names, stats["rows"], color="#36688d")
    rate_ax.barh(names, stats["rate_pct"], color="#b36531")
    rate_ax.axvline(fraud_rate, color="#333333", linestyle="--")
    for ax in (volume_ax, rate_ax):
        ax.invert_yaxis()
    volume_ax.set(title=field, xlabel="Transactions")
    rate_ax.set(xlabel="Fraud share (%)")
plt.show()
for field, stats in category_stats.items():
    display(stats.round(3))

## Transaction patterns over time

The upper line shows how many training transactions were recorded each week. The lower line shows the weekly share labelled fraud. The final week ends at the last retained training event, so it has fewer days of data. These plots describe past patterns. A change can reflect the simulation or the batches that happened to be active that week.

Across the interior weeks, fraud shares ranged from 7.56% to 14.50%. That weekly variation is larger than the differences between levels of the eligible categorical fields.

In [ ]:
week = train[clock].dt.to_period("W-SUN").dt.start_time
weekly = train.groupby(week)[target].agg(rows="size", fraud="sum")
weekly["rate_pct"] = 100 * weekly["fraud"] / weekly["rows"]
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(12, 5.5), layout="constrained")
ax1.plot(weekly.index, weekly["rows"], color="#36688d")
ax2.plot(weekly.index, weekly["rate_pct"], color="#b36531")
ax2.axhline(fraud_rate, linestyle="--", color="#333333")
ax1.set(ylabel="Transactions", title="Weekly training volume")
ax2.set(ylabel="Fraud share (%)", xlabel="Week starting Monday", title="Weekly training fraud share")
plt.show()
display(weekly.tail(8).round(3))

## Numerical readings and fraud

Each dot shows the fraud share in one of ten value bands. The bands have equal width within each field. Larger dots indicate more transactions, and the dashed line is the overall training rate. Sparse bands can show sharp changes from a small number of transactions. Compare the counts in the tables before interpreting a change.

All bands with at least 100 rows had fraud shares between 9.72% and 10.31%.

In [ ]:
bands = {}
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), layout="constrained")
for field, ax in zip(numeric, axes):
    band = pd.cut(train[field], bins=10)
    stats = train.groupby(band, observed=True)[target].agg(rows="size", fraud="sum")
    stats["rate_pct"] = 100 * stats["fraud"] / stats["rows"]
    bands[field] = stats
    midpoint = [interval.mid for interval in stats.index]
    ax.scatter(midpoint, stats["rate_pct"], s=15 + 110 * stats["rows"] / stats["rows"].max(), color="#b36531")
    ax.axhline(fraud_rate, linestyle="--", color="#333333")
    ax.set(title=field, xlabel="Band midpoint", ylabel="Fraud share (%)")
plt.show()
for field, stats in bands.items():
    display(stats.round(3))

## Outlier review

Each box shows the middle half of the values. The line inside is the median. The whiskers reach to observations within 1.5 times the width of the box. The table counts readings outside those limits. Individual outlier dots are hidden in the plots to keep the boxes easy to read. A flag is a reason to check a reading against its context, and the notebook keeps every row.

The rule flagged 5,814 weight readings and no humidity or battery readings.

In [ ]:
outliers = []
fig, axes = plt.subplots(1, 3, figsize=(13, 2.8), layout="constrained")
for field, ax in zip(numeric, axes):
    values = train[field]
    lower_quartile, upper_quartile = values.quantile([.25, .75])
    spread = upper_quartile - lower_quartile
    lower = lower_quartile - 1.5 * spread
    upper = upper_quartile + 1.5 * spread
    below = int(values.lt(lower).sum())
    above = int(values.gt(upper).sum())
    outliers.append((field, lower, upper, below, above, 100 * (below + above) / values.count()))
    sns.boxplot(x=values, color="#89b6cb", showfliers=False, ax=ax)
    ax.set(title=field, xlabel="Recorded value")
plt.show()
outlier_summary = pd.DataFrame(outliers, columns=["field", "lower fence", "upper fence", "below", "above", "flagged_pct"]).set_index("field")
display(outlier_summary.round(4))

## Correlation and leakage screen

The colour grid shows Spearman correlations, which measure whether higher values of one field generally go with higher or lower values of another. Values near zero indicate little consistent rank association. Frauds share labels within batches, so these are descriptive comparisons. Event type has no numeric ordering and is covered by the category charts. The support table checks whether eligible categories or bands with at least 100 rows contain one label only. This is a screen for simple shortcuts in training.

The largest absolute correlation with fraud was about 0.0025. None of the 44 eligible categories or bands with at least 100 rows had only one label.

In [ ]:
correlation_fields = numeric + ["leg", "rfid_scanned", "door_open", target]
correlation = train[correlation_fields].astype(float).corr(method="spearman")
fig, ax = plt.subplots(figsize=(8, 6), layout="constrained")
sns.heatmap(correlation, cmap="vlag", center=0, vmin=-1, vmax=1, annot=True, fmt=".3f", square=True, ax=ax)
ax.set_title("Training Spearman correlations")
plt.show()
screen = []
for field, stats in {**category_stats, **bands}.items():
    supported = stats.loc[stats["rows"].ge(100)]
    single_label = supported["fraud"].eq(0) | supported["fraud"].eq(supported["rows"])
    screen.append((field, len(supported), int(single_label.sum())))
screen_summary = pd.DataFrame(screen, columns=["field", "groups with at least 100 rows", "single-label groups"])
display(screen_summary)

The shortcut review also uses the [feature audit](../reports/feature_audit.csv) and [generator](../Phase1_Dataset/code/generator.py#L238). The code injects signals into seal, QR, GPS dropout and some temperature values using the simulated label. Those fields, label metadata and derived fields with unclear timing stay outside the analysis. The training-only correlation and group checks cannot establish that a feature is safe in a live workflow.

In [ ]:
def markdown_table(frame):
    headers = [str(column) for column in frame.columns]
    def row(values):
        return "| " + " | ".join(str(value) for value in values) + " |"
    return "\n".join([row(headers), row(["---"] * len(headers)), *(row(values) for values in frame.itertuples(index=False, name=None))])


target_correlations = correlation[target].drop(target)
strongest = target_correlations.abs().idxmax()
category_findings = []
for field, stats in category_stats.items():
    low, high = stats["rate_pct"].idxmin(), stats["rate_pct"].idxmax()
    if stats["rate_pct"].nunique() == 1:
        category_findings.append(
            f"- `{field}` has {len(stats)} observed levels. All have the same fraud share of {stats['rate_pct'].iloc[0]:.2f}%."
        )
        continue
    category_findings.append(
        f"- `{field}` has {len(stats)} observed levels. Fraud shares range from "
        f"{stats.loc[low, 'rate_pct']:.2f}% ({low}, {int(stats.loc[low, 'rows']):,} rows) "
        f"to {stats.loc[high, 'rate_pct']:.2f}% ({high}, {int(stats.loc[high, 'rows']):,} rows)."
    )

band_findings = []
for field, stats in bands.items():
    supported = stats.loc[stats["rows"].ge(100)]
    band_findings.append(
        f"- `{field}` has {len(supported)} bands with at least 100 rows. Their fraud shares range "
        f"from {supported['rate_pct'].min():.2f}% to {supported['rate_pct'].max():.2f}%."
    )

# Omit edge weeks from the headline rate range because the split can cut a week short.
interior = weekly.iloc[1:-1]
categorical_shape = pd.DataFrame([
    (field, str(stats["rows"].idxmax()), int(stats["rows"].max()), int(stats["rows"].min()))
    for field, stats in category_stats.items()
], columns=["Field", "Most common level", "Largest level count", "Smallest level count"])
notes = [
    "# Training data exploration",
    f"The [executed notebook](../notebooks/03_Exploratory_Data_Analysis.ipynb) analysed {len(train):,} transactions "
    f"from {len(batch_labels):,} training batches using the fixed split. It used the seven approved current-event "
    "fields, the transaction clock for time summaries, and the separate retrospective target. Scoring-time "
    "versions of conditional lookup fields have yet to be established. [Prediction design](prediction_design.md)",
    "## Target and missing values",
    f"There are {target_counts['Genuine']:,} genuine and {target_counts['Fraud']:,} fraudulent transactions "
    f"({fraud_rate:.2f}% fraud). At batch level, {int(batch_labels.sum()):,} of {len(batch_labels):,} are "
    f"labelled fraud ({batch_labels.mean():.2%}). The eligible fields, clock and target contain "
    f"{int(missing.sum()):,} missing values in total.",
    "## Numerical and categorical distributions",
    markdown_table(numeric_summary[["min", "25%", "50%", "75%", "max", "mean", "std"]].round(3).reset_index(names="Field")),
    markdown_table(categorical_shape),
    "Humidity and battery readings spread fairly evenly across their observed ranges. Weight readings cluster around 10.0.",
    "## Changes over time",
    f"Training events run from {train[clock].min()} through {train[clock].max()}. Across interior weeks, "
    f"weekly volume ranged from {interior['rows'].min():,} to {interior['rows'].max():,} and the fraud share "
    f"ranged from {interior['rate_pct'].min():.2f}% to {interior['rate_pct'].max():.2f}%. The first and last weekly "
    "bins were omitted from these ranges because the training window can cut them short.",
    "## Relationships with fraud",
    "\n".join(category_findings + band_findings),
    f"The largest absolute Spearman correlation with the target among numeric, ordered and Boolean inputs was "
    f"{target_correlations[strongest]:+.4f} for `{strongest}`. The notebook shows the full correlation matrix. "
    "These summaries are descriptive. Rows from the same batch share a label, and differences between categories "
    "or bands may reflect which batches contributed rows.",
    "## Outliers",
    markdown_table(outlier_summary.round(4).reset_index(names="Field")),
    "The fences are descriptive IQR cutoffs. Confirm the reading units and legitimate operating ranges before applying any sensor rule.",
    "## Feature leakage review",
    f"The screen found {screen_summary['single-label groups'].sum():,} single-label categories or bands among "
    f"{screen_summary['groups with at least 100 rows'].sum():,} groups with at least 100 training rows. "
    "This checks simple shortcuts only. Simulator documentation independently supports holding out seal, QR, GPS "
    "dropout, some temperature signals, campaign calendar proxies, target metadata, and undocumented history fields. "
    "Keep batch and transaction identifiers for joining and grouping. [Dataset understanding](dataset_understanding.md#9-synthetic-shortcuts-that-can-reveal-the-target)",
    "## Cleaning and feature engineering decisions for the next phase",
    "- Validate the observed categories and sensor types at input. The training inspection provides no reason to fill values now. Define how a later pipeline handles missing or unseen values when they occur.\n"
    "- Preserve flagged readings unless supporting evidence shows a recording error. Keep rare categories and retain every transaction in the fixed training split.\n"
    "- Use the recorded event type, leg and scan or door flags only if available when scoring. Consider encoding event type as a nominal category and keeping leg as an ordered value. Compare those options on validation once preprocessing begins.\n"
    "- Keep the timestamp for ordering. Any future batch-history feature should use events strictly earlier than the score, with a verified rule for timestamp ties. Keep raw calendar signals outside the initial candidate set.\n"
    "- Keep class counts visible and use fraud-sensitive evaluation metrics later. Fit any encoding, scaling or default values on training only. No transformation was fitted and no data were resampled here.",
    "All graphs and relationship checks use training rows. The original datasets and saved split files were read-only.",
]
report_path = reports / "eda_findings.md"
report_path.write_text("\n\n".join(notes) + "\n", encoding="utf-8")
print(f"Saved {report_path.relative_to(root)}.")